In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Dataset

import pandas as pd

path = "/content/drive/My Drive/Cybersecurity/CIC_IDS2017_400k_Multiclass.csv"
df = pd.read_csv(path)

print(df.head())
print(df.shape)

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0                21             43                  2                       0   
1                80       83913188                  7                       7   
2              3300             56                  1                       1   
3             32784             43                  1                       1   
4                80        1764047                  3                       6   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                           14                            0   
1                          909                        11595   
2                            2                            6   
3                            0                            6   
4                           26                        11607   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                     14                      0            

In [ ]:
import pandas as pd
import numpy as np

# ------------------------------------
# STEP 0: Load dataset
# ------------------------------------
path = "/content/drive/My Drive/Cybersecurity/CIC_IDS2017_400k_Multiclass.csv"
df = pd.read_csv(path)
print("Original dataset shape:", df.shape)

# ------------------------------------
# STEP 1: Label Encoding (BENIGN=0, attacks=1,2,...)
# ------------------------------------
# Get unique attack labels (excluding BENIGN)
attack_labels = df["Label"].unique().tolist()
if "BENIGN" in attack_labels:
    attack_labels.remove("BENIGN")

# Create mapping: BENIGN=0, attacks=1,2,...
label_mapping = {"BENIGN": 0}
for i, label in enumerate(sorted(attack_labels), start=1):
    label_mapping[label] = i

# Apply mapping directly to the 'Label' column
df["Label"] = df["Label"].map(label_mapping).astype(int)

# Print label mapping and counts
print("\nLabel Encoding Mapping:")
for label, code in label_mapping.items():
    print(f"  {label} -> {code}")

print("\nCounts per encoded label:")
print(df["Label"].value_counts().sort_index())

# ------------------------------------
# STEP 2: Keep only numeric features + numeric Label column
# ------------------------------------
numeric_df = df.select_dtypes(include=[np.number]).copy()

# ------------------------------------
# STEP 3: Save final CSV
# ------------------------------------
output_file = "/content/drive/My Drive/Cybersecurity/multiclass_preprocessed_data.csv"
numeric_df.to_csv(output_file, index=False)

print("\nFinal preprocessed dataset saved as:", output_file)
print("Shape:", numeric_df.shape)
numeric_df.head()

Original dataset shape: (606697, 79)

Label Encoding Mapping:
  BENIGN -> 0
  Bot -> 1
  DDoS -> 2
  DoS GoldenEye -> 3
  DoS Hulk -> 4
  DoS Slowhttptest -> 5
  DoS slowloris -> 6
  FTP-Patator -> 7
  Heartbleed -> 8
  Infiltration -> 9
  PortScan -> 10
  SSH-Patator -> 11
  Web Attack � Brute Force -> 12
  Web Attack � Sql Injection -> 13
  Web Attack � XSS -> 14

Counts per encoded label:
Label
0      50000
1       1966
2     128027
3      10293
4     230124
5       5499
6       5796
7       7938
8         11
9         36
10    158930
11      5897
12      1507
13        21
14       652
Name: count, dtype: int64

Final preprocessed dataset saved as: /content/drive/My Drive/Cybersecurity/multiclass_preprocessed_data.csv
Shape: (606697, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,21,43,2,0,14,0,14,0,7.000000,9.899495,...,32,0.0,0.0,0,0,0.0,0.0,0,0,7
1,80,83913188,7,7,909,11595,299,0,129.857143,158.237615,...,20,11984.0,0.0,11984,11984,83300000.0,0.0,83300000,83300000,4
2,3300,56,1,1,2,6,2,2,2.000000,0.000000,...,24,0.0,0.0,0,0,0.0,0.0,0,0,10
3,32784,43,1,1,0,6,0,0,0.000000,0.000000,...,40,0.0,0.0,0,0,0.0,0.0,0,0,10
4,80,1764047,3,6,26,11607,20,0,8.666667,10.263203,...,20,0.0,0.0,0,0,0.0,0.0,0,0,2


In [ ]:
# Split into Train : Val : Test = 60 : 20 : 20

from sklearn.model_selection import train_test_split

X = numeric_df.drop("Label", axis=1)
y = numeric_df["Label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (364018, 78)
Val: (121339, 78)
Test: (121340, 78)


In [ ]:
# Scale Data

import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Replace inf with NaN
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_val.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Option 1: Drop rows with NaN (simplest)
X_train = X_train.dropna()
y_train = y_train[X_train.index]

X_val = X_val.dropna()
y_val = y_val[X_val.index]

X_test = X_test.dropna()
y_test = y_test[X_test.index]

# Option 2: Or you can fill NaN with zero or column mean
# X_train.fillna(0, inplace=True)
# X_val.fillna(0, inplace=True)
# X_test.fillna(0, inplace=True)

# Now scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, "/content/drive/My Drive/Cybersecurity/multiclass_scaler.pkl")

print("Scaling complete")

Scaling complete


In [ ]:
# Reshape for 1D-CNN

X_train_cnn = X_train_scaled.reshape(len(X_train_scaled), X_train_scaled.shape[1], 1)
X_val_cnn = X_val_scaled.reshape(len(X_val_scaled), X_val_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(len(X_test_scaled), X_test_scaled.shape[1], 1)

In [ ]:
# Save Splits to Drive

train_df = pd.concat([pd.DataFrame(X_train_scaled), y_train.reset_index(drop=True)], axis=1)
val_df = pd.concat([pd.DataFrame(X_val_scaled), y_val.reset_index(drop=True)], axis=1)
test_df = pd.concat([pd.DataFrame(X_test_scaled), y_test.reset_index(drop=True)], axis=1)

train_df.to_csv("/content/drive/My Drive/Cybersecurity/multiclass_train_data.csv", index=False)
val_df.to_csv("/content/drive/My Drive/Cybersecurity/multiclass_val_data.csv", index=False)
test_df.to_csv("/content/drive/My Drive/Cybersecurity/multiclass_test_data.csv", index=False)

with open("/content/drive/My Drive/Cybersecurity/multiclass_feature_columns.txt","w") as f:
    f.write("\n".join(list(X.columns)))

In [ ]:
# Build and train 1D-CNN

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

# -----------------------------
# Directory to save everything
# -----------------------------
save_dir = "/content/drive/My Drive/Cybersecurity"
os.makedirs(save_dir, exist_ok=True)

# -----------------------------
# Parameters
# -----------------------------
input_shape = (X_train_cnn.shape[1], 1)
num_classes = len(np.unique(y_train))  # number of classes in your dataset

# -----------------------------
# 1D-CNN model for multiclass
# -----------------------------
model = models.Sequential([
    layers.Conv1D(64, 3, activation="relu", padding="same", input_shape=input_shape),
    layers.Conv1D(128, 3, activation="relu", padding="same"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu", name="embedding_layer"),  # embeddings
    layers.Dense(num_classes, activation="softmax")  # multiclass output
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

# -----------------------------
# Train model
# -----------------------------
model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=20,
    batch_size=32
)

# -----------------------------
# Save trained model
# -----------------------------
model.save(os.path.join(save_dir, "multiclass_model_cnn.h5"))

# -----------------------------
# Generate Train / Validation / Test metrics
# -----------------------------
# Predictions
train_pred = np.argmax(model.predict(X_train_cnn), axis=1)
val_pred   = np.argmax(model.predict(X_val_cnn), axis=1)
test_pred  = np.argmax(model.predict(X_test_cnn), axis=1)

# Classification reports
train_report = classification_report(y_train, train_pred)
val_report   = classification_report(y_val, val_pred)
test_report  = classification_report(y_test, test_pred)

# Confusion matrices
cm_train = confusion_matrix(y_train, train_pred)
cm_val   = confusion_matrix(y_val, val_pred)
cm_test  = confusion_matrix(y_test, test_pred)

# -----------------------------
# Print summaries
# -----------------------------
print("Train Classification Report:\n", train_report)
print("\nValidation Classification Report:\n", val_report)
print("\nTest Classification Report:\n", test_report)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 78, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 15)             │         1,935 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,407 (169.56 KB)

 Trainable params: 43,407 (169.56 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 47s 4ms/step - accuracy: 0.8298 - loss: 0.5560 - val_accuracy: 0.9722 - val_loss: 0.1171
Epoch 2/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 42s 4ms/step - accuracy: 0.9709 - loss: 0.1137 - val_accuracy: 0.9795 - val_loss: 0.0793
Epoch 3/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 43s 4ms/step - accuracy: 0.9782 - loss: 0.0826 - val_accuracy: 0.9827 - val_loss: 0.0604
Epoch 4/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 81s 4ms/step - accuracy: 0.9822 - loss: 0.0651 - val_accuracy: 0.9828 - val_loss: 0.0572
Epoch 5/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 42s 4ms/step - accuracy: 0.9856 - loss: 0.0544 - val_accuracy: 0.9855 - val_loss: 0.0638
Epoch 6/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 42s 4ms/step - accuracy: 0.9872 - loss: 0.0488 - val_accuracy: 0.9902 - val_loss: 0.0392
Epoch 7/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 42s 4ms/step - accuracy: 0.9897 - loss: 0.0404 - val_accuracy: 0.9911 - val_loss: 0.0311
Epoch 8/20
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 44s 4ms/step - accuracy: 

11372/11372 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step
3791/3791 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
3792/3792 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
Train Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.98      0.98     29978
           1       0.89      0.76      0.82      1175
           2       1.00      1.00      1.00     76814
           3       0.98      1.00      0.99      6176
           4       1.00      1.00      1.00    138074
           5       0.99      0.99      0.99      3299
           6       0.99      0.99      0.99      3477
           7       0.99      1.00      1.00      4762
           8       1.00      1.00      1.00         7
           9       0.82      0.41      0.55        22
          10       1.00      1.00      1.00     95270
          11       0.96      0.99      0.98      3538
          12       0.62      0.90      0.73       904
          13       1.00      0.15      0.27        13
          14       1.00    

In [ ]:
# Embedding Model

import tensorflow as tf
from tensorflow.keras import Model, Input
import numpy as np
import os

# -----------------------------
# Paths
# -----------------------------
save_dir = "/content/drive/My Drive/Cybersecurity"
trained_model_path = os.path.join(save_dir, "multiclass_model_cnn.h5")

# -----------------------------
# Load trained multiclass model
# -----------------------------
model = tf.keras.models.load_model(trained_model_path)

# -----------------------------
# Rebuild functional model to output embeddings
# -----------------------------
inp = Input(shape=(X_train_cnn.shape[1], 1))
x = inp

for layer in model.layers:
    x = layer(x)
    if layer.name == "embedding_layer":
        break  # stop at embedding layer

embedding_output = x
embedding_model = Model(inputs=inp, outputs=embedding_output)

embedding_model.summary()

# Save embedding model
embedding_model.save(os.path.join(save_dir, "multiclass_embedding_model_cnn.h5"))

# -----------------------------
# Generate embeddings
# -----------------------------
train_embeddings = embedding_model.predict(X_train_cnn, batch_size=32)
val_embeddings   = embedding_model.predict(X_val_cnn, batch_size=32)
test_embeddings  = embedding_model.predict(X_test_cnn, batch_size=32)

# -----------------------------
# Save embeddings as CSV
# -----------------------------
np.savetxt(os.path.join(save_dir, "multiclass_CIC_IDS2017_train_embeddings.csv"), train_embeddings, delimiter=",")
np.savetxt(os.path.join(save_dir, "multiclass_CIC_IDS2017_val_embeddings.csv"), val_embeddings, delimiter=",")
np.savetxt(os.path.join(save_dir, "multiclass_CIC_IDS2017_test_embeddings.csv"), test_embeddings, delimiter=",")

print("Embedding extraction complete!")

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 78, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 78, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer (Dense)         │ (None, 128)            │        16,512 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,472 (162.00 KB)

 Trainable params: 41,472 (162.00 KB)

 Non-trainable params: 0 (0.00 B)

11372/11372 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step
3791/3791 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
3792/3792 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
Embedding extraction complete!


In [3]:
# Deep Feature Extraction Latency Measurement

import time
import numpy as np
import pandas as pd
import tensorflow as tf

base_path = "/content/drive/My Drive/Cybersecurity/CIC_IDS(2017)_ZeroDay"

# ---------------------------------------------------
# 1️⃣ Load embedding model
# ---------------------------------------------------
embedding_model = tf.keras.models.load_model(
    f"{base_path}/multiclass_embedding_model_cnn.h5"
)

print("Model loaded successfully.")
print("Expected input shape:", embedding_model.input_shape)

# ---------------------------------------------------
# 2️⃣ Load datasets
# ---------------------------------------------------
train_data = pd.read_csv(f"{base_path}/multiclass_train_data.csv")
val_data   = pd.read_csv(f"{base_path}/multiclass_val_data.csv")
test_data  = pd.read_csv(f"{base_path}/multiclass_test_data.csv")

print("Original train shape:", train_data.shape)

# ---------------------------------------------------
# 3️⃣ Remove label column (last column)
# ---------------------------------------------------
X_train = train_data.iloc[:, :-1].values
X_val   = val_data.iloc[:, :-1].values
X_test  = test_data.iloc[:, :-1].values

print("Features shape after label removal:", X_train.shape)

# ---------------------------------------------------
# 4️⃣ Reshape for CNN input (78 features → 78x1)
# ---------------------------------------------------
X_train = X_train.reshape(-1, 78, 1)
X_val   = X_val.reshape(-1, 78, 1)
X_test  = X_test.reshape(-1, 78, 1)

print("Final input shape:", X_train.shape)

# ---------------------------------------------------
# 5️⃣ Measure embedding extraction time
# ---------------------------------------------------
start_time = time.time()

train_emb = embedding_model.predict(X_train, batch_size=32)
val_emb   = embedding_model.predict(X_val, batch_size=32)
test_emb  = embedding_model.predict(X_test, batch_size=32)

end_time = time.time()

total_time = end_time - start_time
total_samples = len(X_train) + len(X_val) + len(X_test)

print("\n Embedding Extraction Complete!")
print("Total Time:", round(total_time, 2), "seconds")
print("Total Samples:", total_samples)
print("Time per Sample:", round(total_time / total_samples, 6), "seconds")

Model loaded successfully.
Expected input shape: (None, 78, 1)
Original train shape: (363900, 79)
Features shape after label removal: (363900, 78)
Final input shape: (363900, 78, 1)
11372/11372 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step
3791/3791 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
3792/3792 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step

 Embedding Extraction Complete!
Total Time: 43.95 seconds
Total Samples: 606521
Time per Sample: 7.2e-05 seconds
